Clases Principales
* Nodo: funciones de creación e identificación
* Grafo: funciones de creación, nodos y aristas

In [ ]:
class Nodo:
  def __init__(self, id):
    self.id = id

  def identificar(self):
    return(f"Nodo id={self.id}")

  def comprobar(self, other):
      if not isinstance(other, Nodo):
          return NotImplemented
      return self.id == other.id


class Arista:
  def __init__(self, origen, destino, distancia=1):
    if not isinstance(origen, Nodo) or not isinstance(destino, Nodo):
        raise ValueError("Origen y destino deben ser instancias de Nodo")
    self.origen = origen
    self.destino = destino
    self.distancia = distancia

  def identificar(self):
    return(f"Arista ({self.origen.id},{self.destino.id}) distancia={self.distancia}")

  def comprobar(self, other):
    if not isinstance(other, Arista):
        return NotImplemented
    return (self.origen == other.origen and self.destino == other.destino and self.distancia == other.distancia) or \
           (self.origen == other.destino and self.destino == other.origen and self.distancia == other.distancia)


class Grafo:
    def __init__(self, dirigido=False):
        #Adyacencia es un diccionario que mapea el ID de un nodo a una lista de aristas que salen de él
        self.nodos = set()
        self.aristas = set()
        self.adyacencia = {} #Diccionario de listas de adyacencia: {id_nodo: [arista1, arista2, ...]}
        self.dirigido = dirigido

    def agregar_nodo(self, nodo):
        if not isinstance(nodo, Nodo):
            raise ValueError("El elemento a agregar debe ser una instancia de Nodo")
        existe = any(nodo.comprobar(n) is True for n in self.nodos)
        if existe:
            raise ValueError("El nodo ya existe en el grafo")
        self.nodos.add(nodo)
        self.adyacencia[nodo.id] = [] #Inicializa su lista de adyacencia vacía

    def agregar_arista(self, arista):
        if not isinstance(arista, Arista):
            raise ValueError("El elemento a agregar debe ser una instancia de Arista")
        #Asegurarse de que los nodos de la arista existan en el grafo. Si no existen, los añade
        if arista.origen not in self.nodos:
            self.agregar_nodo(arista.origen)
        if arista.destino not in self.nodos:
            self.agregar_nodo(arista.destino)
        self.aristas.add(arista)
        self.adyacencia[arista.origen.id].append(arista)
        #Si el grafo no es dirigido
        if not self.dirigido:
            arista_inversa = Arista(arista.destino, arista.origen, arista.distancia)
            self.adyacencia[arista.destino.id].append(arista_inversa)

    def get_nodos(self):
        return list(self.nodos)

    def get_aristas(self):
        return list(self.aristas)

    #Lista de los nodos a los que se puede llegar directamente desde 'nodo_id'.
    def get_vecinos(self, nodo_id):
        if nodo_id not in self.adyacencia:
            raise ValueError(f"El nodo con ID {nodo_id} no existe en el grafo.")
        return [arista.destino for arista in self.adyacencia[nodo_id]]

    def identificar(self):
        return(f"Grafo(numero de nodos={len(self.nodos)}, numero de aristas={len(self.aristas)}, dirigido={self.dirigido})")

    def guardar_graphviz(self, ruta_archivo):
        conector = "->" if self.dirigido else "--"
        tipo_grafo = "digraph" if self.dirigido else "graph"
        lineas = []
        lineas.append(f"{tipo_grafo} G {{")
        #Nodos
        for nodo in sorted(self.get_nodos(), key=lambda n: n.id):
            lineas.append(f"  {nodo.id};")
        #Aristas
        if self.dirigido:
            for arista in self.get_aristas():
                lineas.append(f"  {arista.origen.id} {conector} {arista.destino.id} [label=\"{arista.distancia}\"];")
        else:
            #Para no duplicar aristas bidireccionales en el formato simple no dirigido
            aristas_guardadas = set()
            for arista in self.get_aristas():
                par = tuple(sorted([arista.origen.id, arista.destino.id]))
                if par not in aristas_guardadas:
                    aristas_guardadas.add(par)
                    lineas.append(f"  {arista.origen.id} {conector} {arista.destino.id} [label=\"{arista.distancia}\"];")
        lineas.append("}")
        with open(ruta_archivo, "w", encoding="utf-8") as f:
            f.write("\n".join(lineas) + "\n")

Pruebas para creación de grafos

In [ ]:
nodo1 = Nodo(1)
print(nodo1.identificar())
nodo2 = Nodo(2)
nodo3 = Nodo(3)
arista1 = Arista(nodo1, nodo2, 10)
print(arista1.identificar())

grafo1 = Grafo(True)
grafo1.agregar_nodo(nodo1)
grafo1.agregar_nodo(nodo2)
grafo1.agregar_nodo(nodo3)
grafo1.agregar_arista(arista1)
grafo1.agregar_arista(Arista(nodo1, nodo3, 5))
grafo1.agregar_arista(Arista(nodo3, Nodo(4), 5))
print("Nodos: ", [n.id for n in grafo1.get_nodos()])
print("Aristas:")
for arista in grafo1.get_aristas():
    print(f"  ({arista.origen.id}, {arista.destino.id})")
print(f"Vecinos de ", nodo1.identificar(), " :", [vecino.id for vecino in grafo1.get_vecinos(nodo1.id)])
grafo1.identificar()



Nodo id=1
Arista (1,2) distancia=10
Nodos:  [2, 4, 3, 1]
Aristas:
  (1, 2)
  (3, 4)
  (1, 3)
Vecinos de  Nodo id=1  : [2, 3]


'Grafo(numero de nodos=4, numero de aristas=3, dirigido=True)'

In [ ]:
"""
Genera un grafo completo con N nodos.
    N (int): El número de nodos en el grafo.
    dirigido (bool): True si el grafo debe ser dirigido, False en caso contrario (por defecto).
"""
def generar_grafo_completo(N, dirigido=False):
    grafo = Grafo(dirigido=dirigido)
    nodos = [Nodo(i) for i in range(N)]

    for nodo in nodos:
        grafo.agregar_nodo(nodo)

    #Para un grafo completo, cada par de nodos distintos está conectado por una arista
    if dirigido:
        # Para grafos dirigidos, se añaden aristas en ambas direcciones si no se especifica lo contrario
        for i in range(N):
            for j in range(N):
                if i != j:
                    origen = nodos[i]
                    destino = nodos[j]
                    grafo.agregar_arista(Arista(origen, destino))
    else:
        #Para grafos no dirigidos
        for i in range(N):
            for j in range(i + 1, N):
                origen = nodos[i]
                destino = nodos[j]
                grafo.agregar_arista(Arista(origen, destino))

    return grafo

# Ejemplo de uso:
print("--- Grafo Completo no Dirigido (N=5) ---")
grafo_completo_no_dirigido = generar_grafo_completo(5)
print(grafo_completo_no_dirigido.identificar())
print("Nodos: ", [n.id for n in grafo_completo_no_dirigido.get_nodos()])
print("Aristas:")
for arista in grafo_completo_no_dirigido.get_aristas():
  print(f"  ({arista.origen.id}, {arista.destino.id})")
ruta = "grafo_completo_no_dirigido.gv"
grafo_completo_no_dirigido.guardar_graphviz(ruta)

print("\n--- Grafo Completo Dirigido (N=3) ---")
grafo_completo_dirigido = generar_grafo_completo(3, dirigido=True)
print(grafo_completo_dirigido.identificar())
print("Nodos: ", [n.id for n in grafo_completo_dirigido.get_nodos()])
print("Aristas:")
for arista in grafo_completo_dirigido.get_aristas():
  print(f"  ({arista.origen.id}, {arista.destino.id})")
ruta = "grafo_completo_dirigido.gv"
grafo_completo_dirigido.guardar_graphviz(ruta)

--- Grafo Completo no Dirigido (N=5) ---
Grafo(numero de nodos=5, numero de aristas=10, dirigido=False)
Nodos:  [4, 0, 3, 1, 2]
Aristas:
  (0, 4)
  (1, 4)
  (0, 1)
  (0, 3)
  (0, 2)
  (2, 3)
  (3, 4)
  (1, 2)
  (1, 3)
  (2, 4)

--- Grafo Completo Dirigido (N=3) ---
Grafo(numero de nodos=3, numero de aristas=6, dirigido=True)
Nodos:  [2, 1, 0]
Aristas:
  (2, 0)
  (0, 2)
  (2, 1)
  (0, 1)
  (1, 0)
  (1, 2)


In [ ]:
grafo_ejemplo = generar_grafo_completo(4, dirigido=False)
ruta = "grafo_completo.gv"
grafo_ejemplo.guardar_graphviz(ruta)

print(f"Grafo guardado exitosamente en '{ruta}'. Contenido del archivo:")
with open(ruta, "r") as f:
    print(f.read())

Grafo guardado exitosamente en 'grafo_completo.gv'. Contenido del archivo:
graph G {
  0;
  1;
  2;
  3;
  0 -- 1 [label="1"];
  0 -- 3 [label="1"];
  2 -- 3 [label="1"];
  1 -- 3 [label="1"];
  0 -- 2 [label="1"];
  1 -- 2 [label="1"];
}



Modelos de Generación de Grafos

In [ ]:
import random


"""
Modelos de malla
m: número de columnas (> 1)
n: número de filas (> 1)
"""
def grafoMalla(m, n, dirigido=False):
    if m <= 1 or n <= 1:
        raise ValueError("m y n deben ser mayores que 1")
    grafo = Grafo(dirigido=dirigido)
    #Matriz de nodos para facilitar el acceso por coordenadas (i, j)
    nodos = {}
    for i in range(m):
        for j in range(n):
            #Identificador único para cada nodo en la malla
            nodo_id = f"n_{i}_{j}"
            nodo = Nodo(nodo_id)
            grafo.agregar_nodo(nodo)
            nodos[(i, j)] = nodo
    #Aristas según el modelo de malla
    for i in range(m):
        for j in range(n):
            nodo_actual = nodos[(i, j)]
            #Conectar con el nodo de la siguiente columna (i+1, j)
            if i < m - 1:
                nodo_derecha = nodos[(i + 1, j)]
                grafo.agregar_arista(Arista(nodo_actual, nodo_derecha))
            #Conectar con el nodo de la siguiente fila (i, j+1)
            if j < n - 1:
                nodo_abajo = nodos[(i, j + 1)]
                grafo.agregar_arista(Arista(nodo_actual, nodo_abajo))
    return grafo


"""
Modelo de Erdős-Rényi (G_{n,M}).
n: Número de nodos (> 0)
m: Número de aristas a crear (0 <= m <= n*(n-1)/2 si no dirigido)
"""
def grafoErdosRenyi(n, m, dirigido=False):
    if n <= 0:
        raise ValueError("El número de nodos debe ser mayor a 0")
    grafo = Grafo(dirigido=dirigido)
    nodos = [Nodo(i) for i in range(n)]
    for nodo in nodos:
        grafo.agregar_nodo(nodo)
    #Generar todos los pares posibles de conexiones únicas
    posibles_aristas = []
    for i in range(n):
        for j in range(n):
            if i != j:
                if dirigido or i < j:
                    posibles_aristas.append((nodos[i], nodos[j]))
    if m > len(posibles_aristas):
        raise ValueError(f"Demasiadas aristas solicitadas. El máximo posible para n={n} es {len(posibles_aristas)}")
    #Seleccionar aleatoriamente m aristas
    aristas_seleccionadas = random.sample(posibles_aristas, m)
    for origen, destino in aristas_seleccionadas:
        grafo.agregar_arista(Arista(origen, destino))
    return grafo


"""
Modelo de Gilbert (G_{n,p}).
n: Número de nodos (> 0)
p: Probabilidad de creación de cada arista (0.0 <= p <= 1.0)
"""
def grafoGilbert(n, p, dirigido=False):
    if n <= 0:
        raise ValueError("El número de nodos debe ser mayor a 0")
    if not (0.0 <= p <= 1.0):
        raise ValueError("La probabilidad p debe estar entre 0.0 y 1.0")
    grafo = Grafo(dirigido=dirigido)
    nodos = [Nodo(i) for i in range(n)]
    for nodo in nodos:
        grafo.agregar_nodo(nodo)
    #Recorrer todos los pares y agregarlos basándose en la probabilidad p
    for i in range(n):
        for j in range(n):
            if i != j:
                if dirigido or i < j:
                    if random.random() < p:
                        grafo.agregar_arista(Arista(nodos[i], nodos[j]))
    return grafo


"""
Modelo geográfico simple
n: número de nodos (> 0)
r: distancia máxima para crear una arista (0, 1)
"""
def grafoGeografico(n, r, dirigido=False):
    if n <= 0:
        raise ValueError("El número de nodos debe ser mayor a 0")
    if not (0 <= r <= 1):
        raise ValueError("La distancia r debe estar entre 0 y 1")
    grafo = Grafo(dirigido=dirigido)
    nodos = [Nodo(i) for i in range(n)]
    #Coordenadas aleatorias (x, y) en el plano
    coordenadas = {nodo.id: (random.random(), random.random()) for nodo in nodos}
    for nodo in nodos:
        grafo.agregar_nodo(nodo)
    #Crear aristas si la distancia euclidiana es menor o igual a r
    for i in range(n):
        for j in range(n):
            if i != j:
                if dirigido or i < j:
                    x1, y1 = coordenadas[nodos[i].id]
                    x2, y2 = coordenadas[nodos[j].id]
                    distancia = math.sqrt((x2 - x1)**2 + (y2 - y1)**2)
                    if distancia <= r:
                        grafo.agregar_arista(Arista(nodos[i], nodos[j]))
    return grafo

"""
Modelo Barabasi-Albert
n: número de nodos (> 0)
d: grado máximo esperado por cada nodo (> 1)
"""
def grafoBarabasiAlbert(n, d, dirigido=False):
    if n <= 0:
        raise ValueError("El número de nodos debe ser mayor a 0")
    if d <= 1 or d >= n:
        raise ValueError("d debe ser mayor que 1 y menor que n")
    grafo = Grafo(dirigido=dirigido)
    nodos = [Nodo(i) for i in range(n)]
    #Los primeros d nodos se conectan todos con todos (grafo completo inicial)
    for i in range(d):
        grafo.agregar_nodo(nodos[i])
    for i in range(d):
        for j in range(i + 1, d):
            grafo.agregar_arista(Arista(nodos[i], nodos[j]))
    #Agregar los nodos restantes uno a uno mediante conexión preferencial
    for i in range(d, n):
        nuevo_nodo = nodos[i]
        grafo.agregar_nodo(nuevo_nodo)
        #Calcular grados de todos los nodos actualmente en el grafo
        nodos_existentes = grafo.get_nodos()
        grados = {}
        for n_ext in nodos_existentes:
            if n_ext.id == nuevo_nodo.id:
                continue
            #Grado de un nodo se puede determinar sumando sus conexiones en la lista de adyacencia
            grados[n_ext] = len(grafo.adyacencia.get(n_ext.id, []))
        total_grados = sum(grados.values())
        #Seleccionar d destinos basados en la probabilidad proporcional a su grado
        destinos_seleccionados = []
        intentos = 0
        while len(destinos_seleccionados) < d and intentos < 100:
            intentos += 1
            if total_grados == 0:
                #Si no hay aristas aún, seleccionamos uniformemente al azar
                candidato = random.choice([nd for nd in nodos_existentes if nd.id != nuevo_nodo.id])
            else:
                r_val = random.uniform(0, total_grados)
                acumulado = 0
                candidato = None
                for nd, g in grados.items():
                    acumulado += g
                    if r_val <= acumulado:
                        candidato = nd
                        break
            if candidato and candidato not in destinos_seleccionados:
                destinos_seleccionados.append(candidato)
        for destino in destinos_seleccionados:
            grafo.agregar_arista(Arista(nuevo_nodo, destino))
    return grafo


"""
Modelo Dorogovtsev-Mendes
n: número de nodos (≥ 3)
"""
def grafoDorogovtsevMendes(n, dirigido=False):
    if n < 3:
        raise ValueError("El número de nodos debe ser mayor o igual a 3")
    grafo = Grafo(dirigido=dirigido)
    nodos = [Nodo(i) for i in range(n)]
    #Crear el triángulo inicial con los primeros 3 nodos
    for i in range(3):
        grafo.agregar_nodo(nodos[i])
    grafo.agregar_arista(Arista(nodos[0], nodos[1]))
    grafo.agregar_arista(Arista(nodos[1], nodos[2]))
    grafo.agregar_arista(Arista(nodos[2], nodos[0]))
    #Agregar los nodos restantes seleccionando una arista al azar
    for i in range(3, n):
        nuevo_nodo = nodos[i]
        grafo.agregar_nodo(nuevo_nodo)
        #Elegir una arista al azar del conjunto de aristas registradas
        arista_azar = random.choice(list(grafo.get_aristas()))
        #Conectar el nuevo nodo a ambos extremos de la arista elegida
        grafo.agregar_arista(Arista(nuevo_nodo, arista_azar.origen))
        grafo.agregar_arista(Arista(nuevo_nodo, arista_azar.destino))
    return grafo

In [ ]:
#Pruebas

#Malla
print("--- Grafo de Malla 3x3 No Dirigido ---")
malla_no_dirigida = grafoMalla(3, 3, dirigido=False)
print(malla_no_dirigida.identificar())
print("Nodos creados:", [n.id for n in malla_no_dirigida.get_nodos()])
print("Aristas generadas:")
for arista in malla_no_dirigida.get_aristas():
    print(f"  {arista.origen.id} -- {arista.destino.id}")
ruta_malla = "grafo_malla_3x3.gv"
malla_no_dirigida.guardar_graphviz(ruta_malla)
print(f"\nGrafo de malla exportado exitosamente a '{ruta_malla}'.")

#Erdős-Rényi
print("--- Grafo Erdős-Rényi (n=6, m=8) ---")
grafo_er = grafoErdosRenyi(6, 8, dirigido=False)
print(grafo_er.identificar())
grafo_er.guardar_graphviz("grafo_erdos_renyi.gv")

#Gilbert
print("\n--- Grafo Gilbert (n=6, p=0.45) ---")
grafo_gil = grafoGilbert(6, 0.45, dirigido=False)
print(grafo_gil.identificar())
grafo_gil.guardar_graphviz("grafo_gilbert.gv")
print("Archivos guardados como 'grafo_erdos_renyi.gv' y 'grafo_gilbert.gv'.")

#Geográfico
print("--- Grafo Geográfico (n=10, r=0.3) ---")
grafo_geo = grafoGeografico(10, 0.3, dirigido=False)
print(grafo_geo.identificar())
grafo_geo.guardar_graphviz("grafo_geografico.gv")

#Barabási-Albert
print("\n--- Grafo Barabási-Albert (n=10, d=3) ---")
grafo_ba = grafoBarabasiAlbert(10, 3, dirigido=False)
print(grafo_ba.identificar())
grafo_ba.guardar_graphviz("grafo_barabasi_albert.gv")

#Dorogovtsev-Mendes
print("\n--- Grafo Dorogovtsev-Mendes (n=10) ---")
grafo_dm = grafoDorogovtsevMendes(10, dirigido=False)
print(grafo_dm.identificar())
grafo_dm.guardar_graphviz("grafo_dorogovtsev_mendes.gv")

print("\nNuevos archivos DOT de GraphViz generados para pruebas.")

--- Grafo de Malla 3x3 No Dirigido ---
Grafo(numero de nodos=9, numero de aristas=12, dirigido=False)
Nodos creados: ['n_1_1', 'n_1_2', 'n_2_1', 'n_0_2', 'n_0_0', 'n_1_0', 'n_2_2', 'n_2_0', 'n_0_1']
Aristas generadas:
  n_0_2 -- n_1_2
  n_1_0 -- n_2_0
  n_0_0 -- n_1_0
  n_2_1 -- n_2_2
  n_1_1 -- n_2_1
  n_0_0 -- n_0_1
  n_1_1 -- n_1_2
  n_1_2 -- n_2_2
  n_2_0 -- n_2_1
  n_1_0 -- n_1_1
  n_0_1 -- n_0_2
  n_0_1 -- n_1_1

Grafo de malla exportado exitosamente a 'grafo_malla_3x3.gv'.
--- Grafo Erdős-Rényi (n=6, m=8) ---
Grafo(numero de nodos=6, numero de aristas=8, dirigido=False)

--- Grafo Gilbert (n=6, p=0.45) ---
Grafo(numero de nodos=6, numero de aristas=3, dirigido=False)
Archivos guardados como 'grafo_erdos_renyi.gv' y 'grafo_gilbert.gv'.
--- Grafo Geográfico (n=10, r=0.3) ---
Grafo(numero de nodos=10, numero de aristas=11, dirigido=False)

--- Grafo Barabási-Albert (n=10, d=3) ---
Grafo(numero de nodos=10, numero de aristas=24, dirigido=False)

--- Grafo Dorogovtsev-Mendes (n=10) -

In [ ]:
# Generación de 3 grafos (50, 200 y 500 nodos) por cada modelo

print("Generando grafos del modelo: Malla")
# 50 nodos (5x10), 200 nodos (10x20), 500 nodos (20x25)
grafoMalla(5, 10).guardar_graphviz("malla_50.gv")
grafoMalla(10, 20).guardar_graphviz("malla_200.gv")
grafoMalla(20, 25).guardar_graphviz("malla_500.gv")
print(" -> Guardados malla_50.gv, malla_200.gv, malla_500.gv")

print("Generando grafos del modelo: Erdős-Rényi")
# Elegimos m aristas proporcionales para mantener densidades razonables
grafoErdosRenyi(50, 100).guardar_graphviz("erdos_renyi_50.gv")
grafoErdosRenyi(200, 400).guardar_graphviz("erdos_renyi_200.gv")
grafoErdosRenyi(500, 1000).guardar_graphviz("erdos_renyi_500.gv")
print(" -> Guardados erdos_renyi_50.gv, erdos_renyi_200.gv, erdos_renyi_500.gv")

print("Generando grafos del modelo: Gilbert")
# Probabilidad p inversa al número de nodos para no saturar
grafoGilbert(50, 0.08).guardar_graphviz("gilbert_50.gv")
grafoGilbert(200, 0.02).guardar_graphviz("gilbert_200.gv")
grafoGilbert(500, 0.008).guardar_graphviz("gilbert_500.gv")
print(" -> Guardados gilbert_50.gv, gilbert_200.gv, gilbert_500.gv")

print("Generando grafos del modelo: Geográfico")
# Distancia r ajustada para cada tamaño
grafoGeografico(50, 0.2).guardar_graphviz("geografico_50.gv")
grafoGeografico(200, 0.1).guardar_graphviz("geografico_200.gv")
grafoGeografico(500, 0.06).guardar_graphviz("geografico_500.gv")
print(" -> Guardados geografico_50.gv, geografico_200.gv, geografico_500.gv")

print("Generando grafos del modelo: Barabási-Albert")
# d = 3 aristas por nuevo nodo
grafoBarabasiAlbert(50, 3).guardar_graphviz("barabasi_albert_50.gv")
grafoBarabasiAlbert(200, 3).guardar_graphviz("barabasi_albert_200.gv")
grafoBarabasiAlbert(500, 3).guardar_graphviz("barabasi_albert_500.gv")
print(" -> Guardados barabasi_albert_50.gv, barabasi_albert_200.gv, barabasi_albert_500.gv")

print("Generando grafos del modelo: Dorogovtsev-Mendes")
grafoDorogovtsevMendes(50).guardar_graphviz("dorogovtsev_mendes_50.gv")
grafoDorogovtsevMendes(200).guardar_graphviz("dorogovtsev_mendes_200.gv")
grafoDorogovtsevMendes(500).guardar_graphviz("dorogovtsev_mendes_500.gv")
print(" -> Guardados dorogovtsev_mendes_50.gv, dorogovtsev_mendes_200.gv, dorogovtsev_mendes_500.gv")

Generando grafos del modelo: Malla
 -> Guardados malla_50.gv, malla_200.gv, malla_500.gv
Generando grafos del modelo: Erdős-Rényi
 -> Guardados erdos_renyi_50.gv, erdos_renyi_200.gv, erdos_renyi_500.gv
Generando grafos del modelo: Gilbert
 -> Guardados gilbert_50.gv, gilbert_200.gv, gilbert_500.gv
Generando grafos del modelo: Geográfico
 -> Guardados geografico_50.gv, geografico_200.gv, geografico_500.gv
Generando grafos del modelo: Barabási-Albert
 -> Guardados barabasi_albert_50.gv, barabasi_albert_200.gv, barabasi_albert_500.gv
Generando grafos del modelo: Dorogovtsev-Mendes
 -> Guardados dorogovtsev_mendes_50.gv, dorogovtsev_mendes_200.gv, dorogovtsev_mendes_500.gv
